# Crop recommendation: retrain with the satellite inputs (protocol 1.1)

Team 8, Part B (Thierry). This notebook trains and tests every model of protocol 1.1 (`docs/training_protocol.md`, section 10): the 12 model and input combinations of the midterm, plus 6 that also use four satellite inputs measured at each field. The official results are committed in `01-crop-recommendation/reports/eval_v2`. This notebook runs the same command again and checks that it gives exactly the same numbers.

**What it does** (about 5 minutes on a normal CPU runtime, no GPU needed):

1. Gets the code and the data.
2. Uses the library versions of the official run.
3. Looks at the training data and the four satellite inputs.
4. Trains and tests all 18 combinations with the two tests of the protocol, then checks the results against the committed ones.
5. Shows the scores, what the satellite inputs changed, the scores per district, the comparison with the midterm and the decision.

**How to run it.** On Colab: Runtime, then Run all. On a laptop: open it from `01-crop-recommendation/notebooks` in the repo, with the pinned versions installed (`python -m pip install -r 01-crop-recommendation/requirements-eval.txt`).

In [1]:
# Settings. Change BRANCH only to run code that is not merged into main yet.
import os, subprocess, sys, tempfile
from pathlib import Path

REPO_URL = "https://github.com/ishimwethierryhenry/agri-decision-support-rwanda.git"
BRANCH = "main"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    WORK = Path("/content")
    REPO = WORK / "agri-decision-support-rwanda"
else:
    here = Path.cwd().resolve()
    REPO = next((p for p in [here, *here.parents] if (p / ".git").exists() and (p / "01-crop-recommendation").exists()), None)
    if REPO is None:
        raise SystemExit("Open this notebook from inside the repo folder (01-crop-recommendation/notebooks).")
    WORK = Path(tempfile.mkdtemp(prefix="retrain_v11_"))
MODULE = REPO / "01-crop-recommendation"
OUT = WORK / "eval_v2_check"   # this run only; the official results stay in reports/eval_v2


def run(cmd, cwd=None):
    """Run a command and print its output with short paths. Returns the exit code."""
    r = subprocess.run([str(c) for c in cmd], cwd=cwd, capture_output=True, text=True)
    text = (r.stdout + r.stderr).replace(str(WORK), "<this run>").replace(str(REPO), "<repo>")
    if text.strip():
        print(text.rstrip())
    return r.returncode


print("Running on", "Colab" if IN_COLAB else "a laptop")

Running on a laptop


## 1. Get the repo

On a laptop, the notebook uses the repo folder it was opened from. On Colab, it clones the repo:

1. a public clone (the repo is public at the moment);
2. if the repo is private again: a clone with a GitHub token saved in Colab secrets under the name `GITHUB_TOKEN` (key icon on the left, add the secret, turn on Notebook access). The token is never printed and is removed from the clone right after.

In [2]:
if IN_COLAB:
    import shutil

    def clone(url):
        shutil.rmtree(REPO, ignore_errors=True)
        return subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, url, str(REPO)],
                              capture_output=True, text=True).returncode == 0

    how = "public clone" if clone(REPO_URL) else None
    if how is None:
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
        if token and clone(REPO_URL.replace("https://", f"https://x-access-token:{token}@")):
            subprocess.run(["git", "-C", str(REPO), "remote", "set-url", "origin", REPO_URL], check=True)
            how = "clone with the GITHUB_TOKEN secret"
        token = None
    if how is None:
        raise SystemExit("No clone: the repo may be private and there is no GITHUB_TOKEN secret (see above).")
    print(f"Repo ready ({how}), branch {BRANCH}.")
else:
    print("Using the repo folder this notebook was opened from.")
commit = subprocess.run(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"], capture_output=True, text=True)
print("Code version (commit):", commit.stdout.strip())

Using the repo folder this notebook was opened from.
Code version (commit): a30e114


## 2. The library versions of the official run

Other versions move the numbers: scikit-learn changes the 2 km folds and XGBoost changes its trees. On Colab, the models run in their own environment with the pinned versions of `01-crop-recommendation/requirements-eval.txt` (about one minute). On a laptop, they run with the Python of this notebook, which needs the same versions.

In [3]:
PINNED = {"scikit-learn": "1.7.2", "xgboost": "3.4.1", "pandas": "3.1.0", "numpy": "2.3.5"}
if IN_COLAB:
    VENV = WORK / "venv"
    PY = str(VENV / "bin" / "python")
    run(["pip", "install", "-q", "--root-user-action=ignore", "uv"])
    run(["uv", "venv", "-q", "-p", "3.12", VENV])
    run(["uv", "pip", "install", "-q", "-p", PY, "-r", MODULE / "requirements-eval.txt"])
else:
    PY = sys.executable
check = "import sklearn, xgboost, pandas, numpy; print(sklearn.__version__, xgboost.__version__, pandas.__version__, numpy.__version__)"
got = dict(zip(PINNED, subprocess.run([PY, "-c", check], capture_output=True, text=True).stdout.split()))
for lib, want in PINNED.items():
    print(f"{lib:13s} {got.get(lib, 'missing'):8s}", "ok" if got.get(lib) == want else f"NOT the pinned version {want}")

scikit-learn  1.7.2    ok
xgboost       3.4.1    ok
pandas        3.1.0    ok
numpy         2.3.5    ok


## 3. The training data

Table: `feature_table_v6b.csv`, one row per CIAT field of season B 2025 in Musanze, Nyabihu, Nyagatare and Ruhango. It has the columns of Part A's `feature_table_v5.csv` with the satellite values of `feature_table_v6.csv`, matched on `F_id` (`src/data_processing/build_feature_table_v6b.py`). It has no field coordinates. A field is a training field when its GPS passed Part A's checks (`qc_ok`) and it has one of the four single crops.

**The four new inputs**, measured at each field and known before planting:

- `srtm_elevation_m`: SRTM elevation (30 m);
- `chirps_precip_mm`: CHIRPS rainfall at the field, 1 February to 30 June 2025;
- `ndvi_annual`: Sentinel-2 vegetation index, median of 2024, the year before the season;
- `swir_ratio`: Sentinel-2 shortwave infrared ratio (B11/B8), median of 2024. It is higher on flooded rice fields.

**Not used:** `ndvi_mean` (Sentinel-2 during season B 2025), because satellite data from the same season can show the crop that is already planted.

| View | Inputs |
| --- | --- |
| district | elevation, season weather of the district, temperature at the field's height, NISR crop shares (12) |
| field | elevation, soil 0 to 20 cm (10) |
| field_sat | field, plus the four satellite inputs (14) |
| all | district and field (21) |
| all_sat | all, plus the four satellite inputs (25) |
| all_subsoil | all, plus soil 20 to 50 cm (30, Random Forest only) |

In [4]:
import pandas as pd
from IPython.display import Markdown, display

table = pd.read_csv(MODULE / "data" / "processed" / "feature_table_v6b.csv", low_memory=False)
gps_ok = table["qc_ok"].astype(str).str.strip().str.lower().eq("true")
CROP_LABELS = {"Maize": "Maize", "Beans": "Beans", "Irish potatoes": "Irish potato", "Rice": "Rice"}
train = table[gps_ok & table["Crop_plant"].isin(list(CROP_LABELS))]
print(f"{len(table):,} CIAT fields. {gps_ok.sum():,} pass the GPS checks. "
      f"{len(train):,} of those have one of the four single crops: these are the training fields.")
display(pd.crosstab(train["District"], train["Crop_plant"].map(CROP_LABELS), margins=True, margins_name="Total"))

SAT = {"srtm_elevation_m": "Elevation, SRTM (m)", "chirps_precip_mm": "Rainfall Feb to Jun 2025, CHIRPS (mm)",
       "ndvi_annual": "Vegetation index 2024", "swir_ratio": "SWIR ratio 2024"}
display(Markdown("**The four satellite inputs: median per district**"))
display(train.groupby("District")[list(SAT)].median().rename(columns=SAT).round(2))

2,739 CIAT fields. 2,728 pass the GPS checks. 1,904 of those have one of the four single crops: these are the training fields.


Crop_plant,Beans,Irish potato,Maize,Rice,Total
District,,,,,
Musanze,47,92,24,1,164
Nyabihu,28,103,37,0,168
Nyagatare,237,19,792,36,1084
Ruhango,289,32,157,10,488
Total,601,246,1010,47,1904


**The four satellite inputs: median per district**

,"Elevation, SRTM (m)","Rainfall Feb to Jun 2025, CHIRPS (mm)",Vegetation index 2024,SWIR ratio 2024
District,,,,
Musanze,2103.0,719.21,0.51,0.91
Nyabihu,2397.5,670.10,0.52,0.83
Nyagatare,1416.5,380.23,0.38,1.18
Ruhango,1697.0,520.12,0.41,1.10


## 4. Train and test all 18 combinations

Five kinds of method go through the same folds: B0 (always the most common crop), B1 (expert rules from published crop ranges, not trained on our data), Logistic Regression, Random Forest and XGBoost, the last three on each input view. Settings of the protocol, nothing tuned, seed 0: Random Forest with 300 trees and at least 5 fields per leaf; XGBoost with 300 trees, depth 4 and learning rate 0.05; class-balanced weights for all three.

Two tests:

- **Main test, a district the model never saw:** train on three districts, test on the fourth, four times. It decides whether a model replaces B1.
- **Second test, new fields in the known districts:** 5 folds, and each 2 km block stays on one side, so neighbouring fields are never split between training and test.

This is the command of the official run (2 to 4 minutes). The second cell checks every result file against the committed `reports/eval_v2`.

In [5]:
status = run([PY, MODULE / "src" / "models" / "evaluate.py", "--protocol", "1.1", "--out", OUT], cwd=REPO)
if status != 0:
    raise SystemExit("The evaluation stopped with an error (see above).")

Protocol 1.1. Feature table: <repo>/01-crop-recommendation/data/processed/feature_table_v6b.csv

== main: leave one district out
                 macro_f1_mean  top1_mean  districts_beating_B1 passes_rule
model                                                                      
B1_expert_rules          0.378      0.523                   NaN            
RF[field_sat]            0.368      0.608                   2.0          no
XGB[district]            0.365      0.616                   2.0          no
RF[district]             0.365      0.598                   2.0          no
RF[all_sat]              0.355      0.610                   1.0          no
RF[all_subsoil]          0.354      0.617                   1.0          no
RF[field]                0.338      0.608                   2.0          no
RF[all]                  0.334      0.601                   1.0          no
XGB[field_sat]           0.308      0.506                   1.0          no
XGB[field]               0.308     

In [6]:
status = run([PY, MODULE / "src" / "models" / "compare_results.py", MODULE / "reports" / "eval_v2", OUT], cwd=REPO)
print()
print("Every result file is identical to the committed eval_v2." if status == 0 else
      "Some numbers differ from the committed eval_v2 (see above). Check the library versions of step 2.")

same confusion_counts.csv
same main_blockcv_results.csv
same main_blockcv_summary.csv
same main_lodo_results.csv
same main_lodo_summary.csv
same used_only_blockcv_results.csv
same used_only_blockcv_summary.csv
same used_only_lodo_results.csv
same used_only_lodo_summary.csv

This run reproduces the committed numbers exactly.

Every result file is identical to the committed eval_v2.


## 5. Scores

**Main score:** in each test district, macro-F1 over the crops with at least 5 fields there, then the mean over the four districts. Macro-F1 gives each crop the same weight, so rice and potato count as much as maize. It goes from 0 (always wrong) to 1 (perfect).

**Adoption rule** (written before any result): a model replaces B1 only if, on the main test, its mean is higher than B1's and it beats B1 by more than 0.01 in at least 3 of the 4 districts.

In [7]:
sys.path.insert(0, str(MODULE / "src" / "models"))
import model_scores as ms

S = ms.Scores(OUT)
for test in ms.TESTS:
    display(Markdown(f"**{ms.TEST_NAMES[test]}**"))
    t = S.main_scores(test).drop(columns="model")
    numbers = [c for c in t.columns if pd.api.types.is_numeric_dtype(t[c])]
    display(t.style.format("{:.3f}", subset=numbers).hide(axis="index"))

**A district the model never saw (main test)**

Model,Inputs,Main score,Worst district,Accuracy (mean),Top-3 (mean),Beats B1 in,Passes rule
"B0, most common crop",none,0.104,0.085,0.227,0.985,0 of 4 districts,
"B1, expert rules",rules,0.378,0.090,0.523,0.948,,
Logistic Regression,district (12),0.229,0.010,0.501,0.963,1 of 4 districts,no
Logistic Regression,field (10),0.241,0.103,0.472,0.929,2 of 4 districts,no
Logistic Regression,field + satellite (14),0.248,0.108,0.447,0.929,2 of 4 districts,no
Logistic Regression,all (21),0.254,0.055,0.509,0.958,1 of 4 districts,no
Logistic Regression,all + satellite (25),0.228,0.067,0.500,0.918,1 of 4 districts,no
Random Forest,district (12),0.365,0.235,0.598,0.983,2 of 4 districts,no
Random Forest,field (10),0.338,0.215,0.608,0.991,2 of 4 districts,no
Random Forest,field + satellite (14),0.368,0.229,0.608,0.988,2 of 4 districts,no


**New fields in the four known districts**

Model,Inputs,Main score,Worst district,Accuracy (mean),Top-3 (mean),Beats B1 in,Passes rule
"B0, most common crop",none,0.135,0.085,0.355,0.985,1 of 4 districts,
"B1, expert rules",rules,0.378,0.090,0.523,0.948,,
Logistic Regression,district (12),0.353,0.221,0.612,0.987,2 of 4 districts,no
Logistic Regression,field (10),0.329,0.292,0.595,0.979,2 of 4 districts,no
Logistic Regression,field + satellite (14),0.344,0.290,0.601,0.976,2 of 4 districts,no
Logistic Regression,all (21),0.365,0.265,0.628,0.982,2 of 4 districts,no
Logistic Regression,all + satellite (25),0.389,0.256,0.644,0.982,2 of 4 districts,no
Random Forest,district (12),0.362,0.254,0.626,0.980,2 of 4 districts,no
Random Forest,field (10),0.418,0.295,0.678,0.990,2 of 4 districts,no
Random Forest,field + satellite (14),0.442,0.318,0.697,0.991,2 of 4 districts,no


### What the satellite inputs changed

The same method on the same inputs, without and then with the four satellite inputs.

In [8]:
for test in ms.TESTS:
    display(Markdown(f"**{ms.TEST_NAMES[test]}**"))
    e = S.satellite_effect(test)
    changes = [c for c in e.columns if c.startswith("Change")]
    display(e.style.format("{:.3f}").format("{:+.3f}", subset=changes))

**A district the model never saw (main test)**

,Field,Field + satellite,Change (field),All,All + satellite,Change (all)
Model,,,,,,
Logistic Regression,0.241,0.248,+0.007,0.254,0.228,-0.026
Random Forest,0.338,0.368,+0.030,0.334,0.355,+0.021
XGBoost,0.308,0.308,+0.000,0.294,0.296,+0.002


**New fields in the four known districts**

,Field,Field + satellite,Change (field),All,All + satellite,Change (all)
Model,,,,,,
Logistic Regression,0.329,0.344,+0.015,0.365,0.389,+0.024
Random Forest,0.418,0.442,+0.024,0.424,0.441,+0.017
XGBoost,0.426,0.453,+0.027,0.451,0.456,+0.005


### Main score in each district

The trained models win in the lowlands (Nyagatare, Ruhango), where the rules are weak. The rules win in the highlands (Musanze, Nyabihu), where published crop ranges already work well.

In [9]:
for test in ms.TESTS:
    display(Markdown(f"**{ms.TEST_NAMES[test]}**"))
    display(S.per_district(test).style.format("{:.3f}"))

**A district the model never saw (main test)**

district,Musanze,Nyabihu,Nyagatare,Ruhango
Model,,,,
"B0, most common crop",0.085,0.120,0.090,0.122
"B1, expert rules",0.592,0.645,0.090,0.186
"Logistic Regression, district inputs",0.401,0.295,0.211,0.010
"Logistic Regression, field inputs",0.277,0.310,0.103,0.272
"Logistic Regression, field + satellite inputs",0.278,0.369,0.108,0.237
"Logistic Regression, all inputs",0.268,0.474,0.221,0.055
"Logistic Regression, all + satellite inputs",0.348,0.264,0.232,0.067
"Random Forest, district inputs",0.447,0.518,0.261,0.235
"Random Forest, field inputs",0.405,0.509,0.223,0.215


**New fields in the four known districts**

district,Musanze,Nyabihu,Nyagatare,Ruhango
Model,,,,
"B0, most common crop",0.085,0.120,0.211,0.122
"B1, expert rules",0.592,0.645,0.090,0.186
"Logistic Regression, district inputs",0.461,0.507,0.222,0.221
"Logistic Regression, field inputs",0.305,0.411,0.292,0.306
"Logistic Regression, field + satellite inputs",0.359,0.429,0.297,0.290
"Logistic Regression, all inputs",0.393,0.485,0.319,0.265
"Logistic Regression, all + satellite inputs",0.430,0.470,0.399,0.256
"Random Forest, district inputs",0.448,0.486,0.261,0.254
"Random Forest, field inputs",0.472,0.515,0.390,0.295


## 6. Compared with the midterm

The midterm evaluation (`reports/eval_v1`, 28 October 2026, protocol 1.0) had the 12 combinations without the satellite inputs.

In [10]:
old = ms.Scores(MODULE / "reports" / "eval_v1")
BASELINES = ["B0_majority", "B1_expert_rules"]
rows = []
for test in ms.TESTS:
    a = old.main_scores(test).set_index("model")
    b = S.main_scores(test).set_index("model")
    best_a = a.drop(index=BASELINES)["Main score"].idxmax()
    best_b = b.drop(index=BASELINES)["Main score"].idxmax()
    rows.append({"Test": ms.TEST_NAMES[test],
                 "Rules": f"{b.loc['B1_expert_rules', 'Main score']:.3f}",
                 "Best model at the midterm": f"{ms.short_label(best_a)}: {a.loc[best_a, 'Main score']:.3f}",
                 "Best model now": f"{ms.short_label(best_b)}: {b.loc[best_b, 'Main score']:.3f}",
                 "Change": f"{round(b.loc[best_b, 'Main score'], 3) - round(a.loc[best_a, 'Main score'], 3):+.3f}"})
display(pd.DataFrame(rows).set_index("Test"))


def unchanged(test):
    a = old.main_scores(test).set_index("model")["Main score"]
    b = S.main_scores(test).set_index("model")["Main score"].reindex(a.index)
    return bool((a - b).abs().max() == 0)


print("The 12 combinations of the midterm give exactly the same scores as on 28 October:",
      "yes" if all(unchanged(t) for t in ms.TESTS) else "no")

,Rules,Best model at the midterm,Best model now,Change
Test,,,,
A district the model never saw (main test),0.378,"XGBoost, district inputs: 0.365","Random Forest, field + satellite inputs: 0.368",+0.003
New fields in the four known districts,0.378,"XGBoost, all inputs: 0.451","XGBoost, all + satellite inputs: 0.456",+0.005


The 12 combinations of the midterm give exactly the same scores as on 28 October: yes


## 7. Decision

In [11]:
main = S.main_scores("new_district").set_index("model")
fields = S.main_scores("new_fields").set_index("model")
passing = main.index[main["Passes rule"] == "yes"].tolist()
best_main = main.drop(index=BASELINES)["Main score"].idxmax()
best_fields = fields.drop(index=BASELINES)["Main score"].idxmax()
if passing:
    print("Passes the adoption rule on the main test:", ", ".join(ms.short_label(m) for m in passing))
else:
    print("No model passes the adoption rule, so the expert rules (B1) stay the default answer of the API.")
    print(f"Closest on a district never seen: {ms.short_label(best_main)}, {main.loc[best_main, 'Main score']:.3f} "
          f"against {main.loc['B1_expert_rules', 'Main score']:.3f} for the rules, ahead by more than 0.01 in "
          f"{main.loc[best_main, 'Beats B1 in']}.")
print(f"Best on new fields in the known districts: {ms.short_label(best_fields)}, "
      f"{fields.loc[best_fields, 'Main score']:.3f} against {fields.loc['B1_expert_rules', 'Main score']:.3f} for the rules.")

No model passes the adoption rule, so the expert rules (B1) stay the default answer of the API.
Closest on a district never seen: Random Forest, field + satellite inputs, 0.368 against 0.378 for the rules, ahead by more than 0.01 in 2 of 4 districts.
Best on new fields in the known districts: XGBoost, all + satellite inputs, 0.456 against 0.378 for the rules.
